# Assignment Title: Analyzing Customer Behavior for E-commerce Insights
Objective: Your task is to analyze a dataset representing customer activity
on an e-commerce platform. The goal is to extract actionable insights that
could help the business improve sales, enhance customer engagement,
and personalize the shopping experience.

## Section 1: Imports and Data Loading


In [ ]:
# ── CELL 1: Imports and Data Loading ───────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# 1. Get the exact absolute folder path where this script is saved
NOTEBOOK_DIR = Path.cwd()

# 2. Point directly to the "data" folder where your CSV files live
DATA_FOLDER = NOTEBOOK_DIR / "data-bin"

# ── Read from CSV ──────────────────────────────────────────────────
# Combine DATA_FOLDER with the file names using the / operator to create robust paths
customers = pd.read_csv(DATA_FOLDER / 'customers.csv')
products  = pd.read_csv(DATA_FOLDER / 'products.csv')

# parse_dates tells Pandas to convert these columns from text strings into true datetime objects
browsing  = pd.read_csv(DATA_FOLDER / 'browsing.csv',  parse_dates=['browse_date'])
purchases = pd.read_csv(DATA_FOLDER / 'purchases.csv', parse_dates=['purchase_date'])

print("All datasets successfully loaded from the data directory!")

## Section 2: EDA

In [ ]:
# ── CELL 2: Exploratory Data Analysis ──────────────────────────────

# Dataset structure overview
print("=== CUSTOMER DATA TYPES ===")
print(customers.dtypes)

# Statistical summary of numerical columns
print("\n=== DESCRIPTIVE STATISTICS ===")
print(customers[['age']].describe())
print(purchases[['total_amount', 'quantity']].describe())

# Missing values heatmap
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sns.heatmap(customers.isnull(), cbar=False, cmap='viridis',
            yticklabels=False, ax=axes[0])
axes[0].set_title('Missing Values — Customers')

# Age distribution
axes[1].hist(customers['age'].dropna(), bins=25, color='cyan',
           edgecolor='black', alpha=0.8)
axes[1].set_title('Customer Age Distribution')
axes[1].set_xlabel('Age')
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# Sales by product category
sales = purchases.merge(products[['product_id', 'category']], on='product_id')
category_rev = sales.groupby('category')['total_amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
category_rev.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Total Revenue by Product Category')
plt.ylabel('Revenue (GHS)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('category_revenue.png', dpi=150)
plt.show()

## Section 3: Data Cleaning

In [ ]:
# ── CELL 3: Data Cleaning ──────────────────────────────────────────

# Step 1: Remove duplicates
print(f"Customer rows before: {len(customers)}")
customers.drop_duplicates(inplace=True)
print(f"Customer rows after:  {len(customers)}")

# print(f"Purchases row before: {len(purchases)}")
purchases.drop_duplicates(subset='transaction_id', inplace=True)
# print(f"Purchases row after: {len(purchases)}")

# Step 2: Handle missing values
# Age: fill with median (robust to outliers)
customers['age'].fillna(customers['age'].median(), inplace=True)
# Device: fill with 'Unknown' category
customers['preferred_device'].fillna('Unknown', inplace=True)

# Step 3: Remove impossible values
purchases = purchases[purchases['total_amount'] > 0]
purchases = purchases[purchases['quantity'] > 0]

# Step 4: Verify data types
purchases['purchase_date'] = pd.to_datetime(purchases['purchase_date'])
browsing['browse_date']   = pd.to_datetime(browsing['browse_date'])
customers['signup_date']  = pd.to_datetime(customers['signup_date'])

print("\nCleaning complete. No missing values remain:")
print(customers.isnull().sum())

## Section 4: Feature Engineering

In [ ]:
# ── CELL 4: Feature Engineering ────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

# ── RFM Features (Recency, Frequency, Monetary) ────────────────────
# Recency  = days since last purchase (lower = more engaged)
# Frequency = how many times they bought
# Monetary  = total money spent

# Establish a baseline snapshot date (1 day after the latest purchase)
snapshot_date = purchases['purchase_date'].max() + pd.Timedelta(days=1)
# print(snapshot_date)

rfm = purchases.groupby('customer_id').agg(
    Recency   = ('purchase_date', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('transaction_id', 'count'),
    Monetary  = ('total_amount', 'sum')
).reset_index()

rfm['avg_order_value'] = rfm['Monetary'] / rfm['Frequency']

# ── Browsing Behavior Features ──────────────────────────────────────
browse_feats = browsing.groupby('customer_id').agg(
    total_browse_time    = ('time_spent_seconds', 'sum'),
    avg_pages_per_session = ('pages_viewed', 'mean'),
    cart_adds            = ('added_to_cart', 'sum'),
    browse_sessions      = ('browse_date', 'count')
).reset_index()

# ── Return Rate Per Customer ────────────────────────────────────────
return_rate = purchases.groupby('customer_id').agg(
    total_orders  = ('transaction_id', 'count'),
    total_returns = ('returned', 'sum')
).reset_index()
return_rate['return_rate'] = return_rate['total_returns'] / return_rate['total_orders']

# ── Time-Based Features from Signup Date ───────────────────────────
customers['days_since_signup'] = (pd.Timestamp.today() - customers['signup_date']).dt.days
customers['signup_month']      = customers['signup_date'].dt.month

# ── Merge All Features into Master Table ───────────────────────────
master = rfm.merge(browse_feats, on='customer_id', how='left') \
             .merge(return_rate[['customer_id', 'return_rate']], on='customer_id', how='left') \
             .merge(customers[['customer_id', 'age', 'gender', 'region',
                                'preferred_device', 'loyalty_tier',
                                'days_since_signup']], on='customer_id', how='left')

# ── Type-Safe Handling of Missing Values (FIX) ────────────────────
# Map appropriate placeholders matching each column's strict data type
# Create a dictionary specifying what to fill for each data type
fill_values = {}
for col in master.columns:
    if master[col].dtype == 'object' or isinstance(master[col].dtype, pd.StringDtype):
        fill_values[col] = "Unknown"  # Fill text columns with a string
    else:
        fill_values[col] = 0          # Fill numeric columns with zero

master.fillna(value=fill_values, inplace=True)
# master.fillna(0, inplace=True)

# ── Encode Categorical Columns ──────────────────────────────────────
# Convert text category words into unique numerical codes for machine learning
le = LabelEncoder()
for col in ['gender', 'region', 'preferred_device', 'loyalty_tier']:
    master[col + '_enc'] = le.fit_transform(master[col].astype(str))

# ── Define Churn Label: Recency > 90 days = churned ────────────────
# Justification: industry standard is 60-120 days; 90 is a balance
master['churned'] = (master['Recency'] > 90).astype(int)

print(f"Master feature table: {master.shape}")
print(f"Churn rate: {master['churned'].mean():.2%}") # Converts it to a percentage in 2 d.p.
master.head()

## Section 5: Predictive Modeling

In [ ]:
# ── CELL 5: Predictive Modeling ────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                               ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
import shap

# ── Define Features ─────────────────────────────────────────────────
FEATURES = ['Recency', 'Frequency', 'Monetary', 'avg_order_value',
            'total_browse_time', 'avg_pages_per_session', 'cart_adds',
            'browse_sessions', 'return_rate', 'age',
            'days_since_signup', 'gender_enc', 'region_enc',
            'preferred_device_enc', 'loyalty_tier_enc']

X = master[FEATURES]
y = master['churned']

# ── Train-Test Split (stratified to preserve churn ratio) ───────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ── Train Random Forest ─────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',  # handles class imbalance
    random_state=42
)
rf.fit(X_train, y_train)

# ── Evaluate ────────────────────────────────────────────────────────
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

# ── 5-Fold Cross-Validation ──────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')
print(f"\n5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ── Confusion Matrix ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=ax,
    display_labels=['Active', 'Churned'], cmap='Blues'
)
plt.title('Confusion Matrix — Churn Prediction')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

# ── Feature Importance ──────────────────────────────────────────────
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Feature Importance — Top Churn Drivers')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

# ── SHAP Explainability ──────────────────────────────────────────────
# SHAP shows WHY the model predicted churn for each customer
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values[:, :, 1], X_test, feature_names=FEATURES, show=False)
plt.title('SHAP Summary — Churn Prediction Drivers')
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6: Big Data Tool — Kafka

In [ ]:
# ── CELL 6: Big Data Tool — Apache Kafka Simulation ────────────────
#
# In production: Browser/App → Kafka Topic → ML Consumer → Dashboard
#
# Topics we would create in a real Kafka deployment:
#   ecommerce.browsing   → real-time page view events
#   ecommerce.purchases  → transaction stream
#   ecommerce.alerts     → churn risk alerts pushed to marketing
#
# We simulate the producer-consumer architecture below.

import json
from collections import deque
import random

class KafkaSimulator:
    """Simulates Kafka topic: produce events, consume in batches."""
    def __init__(self, topic):
        self.topic = topic
        self.queue = deque()

    def produce(self, event: dict):
        event['kafka_timestamp'] = str(pd.Timestamp.now())
        self.queue.append(event)

    def consume(self, n=5):
        consumed = []
        for _ in range(min(n, len(self.queue))):
            consumed.append(self.queue.popleft())
        return consumed

# ── Browsing Stream ──────────────────────────────────────────────────
browse_stream = KafkaSimulator(topic='ecommerce.browsing')

for _ in range(10):
    event = {
        'customer_id': f'C{np.random.randint(1, 2001):04d}',
        'product_id':  f'P{np.random.randint(1, 201):04d}',
        'action': random.choice(['view', 'add_to_cart', 'checkout']),
        'session_duration_s': np.random.randint(10, 300)
    }
    browse_stream.produce(event)

print(f"📡 Consuming 5 events from Kafka topic: '{browse_stream.topic}'")
print("-" * 55)
for msg in browse_stream.consume(5):
    print(json.dumps(msg, indent=2))

# ── Real-Time Churn Alerts ───────────────────────────────────────────
# In production: ML model scores each event and pushes to alert topic
alert_stream = KafkaSimulator(topic='ecommerce.churn_alerts')

high_risk = master[master['churned'] == 1].head(5)

print("\nChurn Alert Stream (ecommerce.churn_alerts topic):")
for _, row in high_risk.iterrows():
    alert = {
        'alert_type': 'HIGH_CHURN_RISK',
        'customer_id': row['customer_id'],
        'days_since_purchase': int(row['Recency']),
        'lifetime_value_ghs': round(row['Monetary'], 2),
        'recommended_action': 'Send 15% discount coupon via SMS'
    }
    alert_stream.produce(alert)
    print(json.dumps(alert, indent=2))

## Section 7: Insights & Visualizations

In [ ]:
# ── CELL 7: Business Insights & Visualizations ─────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# ── Chart 1: Monthly Revenue Trend ──────────────────────────────────
purchases['year_month'] = purchases['purchase_date'].dt.to_period('M')
monthly = purchases.groupby('year_month')['total_amount'].sum()
axes[0,0].plot(monthly.index.astype(str), monthly.values,
               marker='o', color='steelblue', linewidth=2)
axes[0,0].set_title('Monthly Revenue Trend', fontweight='bold')
axes[0,0].set_ylabel('Revenue (GHS)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)

# ── Chart 2: Customer Loyalty Tier ──────────────────────────────────
tier_counts = customers['loyalty_tier'].value_counts()
axes[0,1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%',
             colors=['#CD7F32', '#C0C0C0', '#FFD700', '#E5E4E2'],
             startangle=140)
axes[0,1].set_title('Loyalty Tier Distribution', fontweight='bold')

# ── Chart 3: RFM Scatter (Churn Segmentation) ───────────────────────
sc = axes[0,2].scatter(master['Frequency'], master['Monetary'],
                      c=master['churned'], cmap='RdYlGn_r',
                      alpha=0.5, edgecolors='k', linewidths=0.2)
plt.colorbar(sc, ax=axes[0,2], label='Churned (1=Yes)')
axes[0,2].set_title('Frequency vs Lifetime Value (Churn)', fontweight='bold')
axes[0,2].set_xlabel('Purchase Frequency')
axes[0,2].set_ylabel('Total Spend (GHS)')

# ── Chart 4: Payment Method Popularity ──────────────────────────────
pay_counts = purchases['payment_method'].value_counts()
axes[1,0].barh(pay_counts.index, pay_counts.values,
              color=['#00e5ff', '#7c3aed', '#10b981'], edgecolor='black')
axes[1,0].set_title('Payment Method Preference', fontweight='bold')
axes[1,0].set_xlabel('Number of Transactions')

# ── Chart 5: Churn Rate by Region ────────────────────────────────────
churn_region = master.merge(customers[['customer_id', 'region']],
                              on='customer_id', how='left')
region_churn = churn_region.groupby('region_y')['churned'].mean().sort_values(ascending=False)
axes[1,1].bar(region_churn.index, region_churn.values,
             color='salmon', edgecolor='black')
axes[1,1].set_title('Churn Rate by Region', fontweight='bold')
axes[1,1].set_ylabel('Churn Rate')
axes[1,1].tick_params(axis='x', rotation=30)

# ── Chart 6: Revenue by Loyalty Tier ─────────────────────────────────
tier_rev = purchases.merge(customers[['customer_id', 'loyalty_tier']],
                            on='customer_id', how='left')
tier_rev = tier_rev.groupby('loyalty_tier')['total_amount'].sum()
axes[1,2].bar(tier_rev.index, tier_rev.values,
             color=['#CD7F32', '#C0C0C0', '#FFD700', '#E5E4E2'],
             edgecolor='black')
axes[1,2].set_title('Revenue by Loyalty Tier', fontweight='bold')
axes[1,2].set_ylabel('Revenue (GHS)')

plt.suptitle('E-Commerce Customer Insights Dashboard',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('insights_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()